In [1]:
import numpy as np

import tensorflow as tf

import pandas as pd

In [2]:
np.random.seed(42)
tf.random.set_seed(42)

print('TensorFlow 버전: ', tf.__version__)
print('NumPy 버전: ', np.__version__)
print('Pandas 버전: ', pd.__version__)

TensorFlow 버전:  2.10.0
NumPy 버전:  1.23.5
Pandas 버전:  2.3.3


In [3]:
H_example = np.array([[-3.0], [-1.0], [0.0], [1.0], [3.0]])

H_example

array([[-3.],
       [-1.],
       [ 0.],
       [ 1.],
       [ 3.]])

In [4]:
z_example = tf.sigmoid(H_example)

z_example

<tf.Tensor: shape=(5, 1), dtype=float64, numpy=
array([[0.04742587],
       [0.26894142],
       [0.5       ],
       [0.73105858],
       [0.95257413]])>

In [5]:
sigmoid_result_df = pd.DataFrame({
    'H': H_example.reshape(-1),
    'z = sigmoid(H)': z_example.numpy().reshape(-1)
})

sigmoid_result_df

,H,z = sigmoid(H)
0,-3.0,0.047426
1,-1.0,0.268941
2,0.0,0.500000
3,1.0,0.731059
4,3.0,0.952574


In [6]:
prediction_example = tf.cast(z_example >= 0.5, tf.int32)

sigmoid_result_df['prediction'] = prediction_example.numpy().reshape(-1)

sigmoid_result_df

,H,z = sigmoid(H),prediction
0,-3.0,0.047426,0
1,-1.0,0.268941,0
2,0.0,0.500000,1
3,1.0,0.731059,1
4,3.0,0.952574,1


In [7]:
tf.random.set_seed(42)

model = tf.keras.Sequential([
    tf.keras.Input(shape = (1, )),
    tf.keras.layers.Dense(1)
])

model.summary

<bound method Model.summary of <keras.engine.sequential.Sequential object at 0x000002DE7A3F4820>>

In [8]:
X = np.array([160, 170, 180, 190], dtype = np.float32).reshape(-1, 1)

y = np.array([0, 0, 1, 1], dtype = np.float32).reshape(-1, 1)

print('입력값 X: \n', X)
print('입력값 y: \n', y)
print('X shape:', X.shape)
print('y shape:', y.shape)

입력값 X: 
 [[160.]
 [170.]
 [180.]
 [190.]]
입력값 y: 
 [[0.]
 [0.]
 [1.]
 [1.]]
X shape: (4, 1)
y shape: (4, 1)


In [9]:
X_mean = X.mean()
X_std = X.std()

X_norm = (X - X_mean) / X_std

print('입력값 평균 X_mean: ', X_mean)
print('입력값 표준편차 X_std: ', X_std)
print('정규화된 입력값 X_norm: \n: ', X_norm)
print('X_norm shape: ', X_norm.shape)

입력값 평균 X_mean:  175.0
입력값 표준편차 X_std:  11.18034
정규화된 입력값 X_norm: 
:  [[-1.3416408]
 [-0.4472136]
 [ 0.4472136]
 [ 1.3416408]]
X_norm shape:  (4, 1)


In [10]:
H_before = model(X_norm)

z_before = tf.sigmoid(H_before)

print('학습 전 H 예시:')
print(H_before.numpy()[:5])

print('\n학습 전 sigmoid(H), 즉 z 예시:')
print(z_before.numpy()[:5])

학습 전 H 예시:
[[ 0.17805792]
 [ 0.05935264]
 [-0.05935264]
 [-0.17805792]]

학습 전 sigmoid(H), 즉 z 예시:
[[0.54439723]
 [0.5148338 ]
 [0.4851662 ]
 [0.45560277]]


In [11]:
learning_rate = 0.1

epochs = 1000

model.compile(
    optimizer = tf.keras.optimizers.SGD(learning_rate = learning_rate),
    loss = tf.keras.losses.BinaryCrossentropy(from_logits = True)
)

print('learning_rate: ', learning_rate)
print('epochs: ', epochs)
print('compile 완료: optimizer = SGD, loss = BinaryCrossentropy(from_logits = True)')

learning_rate:  0.1
epochs:  1000
compile 완료: optimizer = SGD, loss = BinaryCrossentropy(from_logits = True)


In [12]:
history = model.fit(
    X_norm,
    y,
    epochs = epochs,
    verbose = 0
)

print('학습 완료!')

print('history에 기록된 항목: ', list(history.history.keys()))

학습 완료!
history에 기록된 항목:  ['loss']


In [13]:
for epoch_index, mean_cost in enumerate(history.history['loss']):
    if epoch_index % 100 == 0 or epoch_index == epochs - 1:
        print(f'epoch = {epoch_index}, mean_cost = {mean_cost:.6f}')
        
final_mean_cost = history.history['loss'][-1]
print(f'\n최종 mean_cost(평균 Cost): {final_mean_cost:.6f}')

epoch = 0, mean_cost = 0.754699
epoch = 100, mean_cost = 0.203917
epoch = 200, mean_cost = 0.135857
epoch = 300, mean_cost = 0.105344
epoch = 400, mean_cost = 0.086946
epoch = 500, mean_cost = 0.074328
epoch = 600, mean_cost = 0.065024
epoch = 700, mean_cost = 0.057835
epoch = 800, mean_cost = 0.052096
epoch = 900, mean_cost = 0.047398
epoch = 999, mean_cost = 0.043513

최종 mean_cost(평균 Cost): 0.043513


In [14]:
H = model(X_norm)

probability = tf.sigmoid(H)

print('학습 후 예시: ')
print(H.numpy()[:5])

print('학습 후 probability(=sigmoid(H), 즉 z) 예시: ')
print(probability.numpy()[:5])

print('\n실제 정답 y: ')
print(y[:5])

학습 후 예시: 
[[-7.2221265]
 [-2.4073753]
 [ 2.4073753]
 [ 7.2221265]]
학습 후 probability(=sigmoid(H), 즉 z) 예시: 
[[7.2971499e-04]
 [8.2612015e-02]
 [9.1738802e-01]
 [9.9927026e-01]]

실제 정답 y: 
[[0.]
 [0.]
 [1.]
 [1.]]


In [15]:
input_value = 185

input_norm = (input_value - X_mean) / X_std

input_norm_array = np.array([[input_norm]], dtype =np.float32)

H_new = model(input_norm_array)

probability = tf.sigmoid(H_new)

In [16]:
prediction = 1 if probability.numpy().item() >= 0.5 else 0

print('새 입력값: ', input_value)
print('정규화된 입력값: ', input_norm)
print('H_new: ', H_new.numpy().item())
print('probability: ', probability.numpy().item())
print('prediction: ', prediction)

print(f'\n키가 {input_value}cm인 사람의 농구선수 확률(probability): {probability.numpy().item():.4f}')
if prediction == 1:
    print('판별 결과: 농구선수입니다.')
else:
    print('판별 결과: 농구선수가 아닙니다.')

새 입력값:  185
정규화된 입력값:  0.8944271969412381
H_new:  4.814750671386719
probability:  0.9919559955596924
prediction:  1

키가 185cm인 사람의 농구선수 확률(probability): 0.9920
판별 결과: 농구선수입니다.


### 개념 체크리스트
- X와 X_norm의 차이는 무엇입니까?
    X는 일반 입력값, X_norm은 일반 입력값에 대해 정규화를 진행한 값입니다.
- H는 확룰입니까, 선형 출력값입니까?
    H는 선형 출력값입니다.
- z = sigmoid(H)는 무엇을 의미합니까?
    선형 출력값 H에 대하여 확률값을 부여하기 위해 시그모이드를 적용하는 과정을 의미합니다.
- Dense(1)에 activation = 'sigmoid'를 넣지 않는 이유는 무엇입니까?
    뒤에서 BinaryCrossentropy(from_logits=True)가 내부적으로 처리를 하기 때문입니다.

### PyTorch와 TensorFlow2 비교 체크리스트
- PyTorch의 BCEWithLogitsLoss는 TensorFlow2에서 무엇에 대응합니까?
    TensorFlow2에서는 BinaryCrossentropy(from_logits = True)와 대응합니다.
- PyTorch의 train loop는 TensorFlow2에서 무엇에 대응합니까?
    TensorFlow2에서는 model.fit()과 대응합니다.
- torch.nn.Linear(1,1)은 TensorFlow2에서 무엇에 대응합니까?
    TensorFlow2에서는 tf.keras.layers.Dense(1)과 대응합니다.
- model.fit() 내부에서는 어떤 일이 반복됩니까?
    설정된 epoch만큼 H 계산 및 Loss 계산, 가중치(w, b) 업데이트 과정을 반복합니다.

### 예측 단계 체크리스트
- 새 입력값 X_new는 왜 정규화해야 합니까?
    분석 모델을 통한 예측 결과를 도출하기 위해서는 학습 데이터와 동일한 조건의 가공 과정을 거쳐야 하기 때문입니다.
- 새 입력값을 정규화할 때 어떤 X_mean, X_std를 사용해야 합니까?
    모델 학습과정에서 사용한 기존 데이터에 대한 평균 및 표준편차를 사용해야 합니다.
- H_new와 z_new의 차이는 무엇입니까?
    H_new는 새로운 입력 데이터에 대한 선형 결과값, z_new는 새로운 입력 데이터의 선형 결과값에 대해 sigmoid를 취한 값입니다.
- 최종 prediction은 어떤 기준으로 결정합니까?
    z_new에 대한 확률 값이 0.5이상이면 해당(1), 0.5 미만이면 비해당(0)으로 결정합니다.